# 01 — Dataset Preparation on Colab
Loads ViMD parquet shards from Google Drive, validates audio, extracts features, applies augmentation, splits into train/val/test, and exports to HuggingFace Dataset format.

In [ ]:
# ── Mount Google Drive ──────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

PROJECT_ROOT = '/content/drive/MyDrive/VietSpeech'
VIMD_PATH = f'{PROJECT_ROOT}/ViMD'
PROCESSED_PATH = f'{PROJECT_ROOT}/processed'
DATASET_EXPORT = f'{PROJECT_ROOT}/hf_dataset'

import sys, os
sys.path.insert(0, PROJECT_ROOT)
print('Drive mounted. PROJECT_ROOT:', PROJECT_ROOT)

In [ ]:
# ── Install additional dependencies ─────────────────────────────────────
!pip install -q soundfile librosa noisereduce pyloudnorm pyworld datasets
!pip install -q torch-audiomentations audiomentations
print('Dependencies installed')

In [ ]:
# ── Load ViMD parquet shards ─────────────────────────────────────────────
import pandas as pd
import glob

parquet_files = sorted(glob.glob(f'{VIMD_PATH}/**/*.parquet', recursive=True))
print(f'Found {len(parquet_files)} parquet shards')

dfs = []
for f in parquet_files[:5]:  # Preview first 5 shards
    df = pd.read_parquet(f)
    dfs.append(df)
    print(f'  {os.path.basename(f)}: {len(df)} rows, columns={list(df.columns)}')

sample_df = pd.concat(dfs, ignore_index=True)
print(f'\nTotal sample rows: {len(sample_df)}')
sample_df.head(3)

In [ ]:
# ── Run ViMD loader ──────────────────────────────────────────────────────
from dataset.ingestion.vimd_loader import ViMDLoader

loader = ViMDLoader(
    data_dir=VIMD_PATH,
    output_dir=f'{PROCESSED_PATH}/audio',
    target_sr=16000,
)
df_all = loader.load_all(max_shards=None)
print(f'Loaded {len(df_all)} utterances from ViMD')
df_all.head()

In [ ]:
# ── Audio Validation ─────────────────────────────────────────────────────
from dataset.validation.audio_validator import AudioValidator
from dataset.validation.label_validator import LabelValidator

audio_val = AudioValidator(target_sr=16000, min_duration=0.5, max_duration=30.0)
label_val = LabelValidator()

audio_report = audio_val.validate_dataframe(df_all)
label_report = label_val.validate(df_all)

print('Audio validation report:')
print(audio_report.summary())
print('\nLabel validation report:')
print(label_report.summary())

# Keep only valid samples
df_valid = audio_report.valid_df
print(f'\nValid samples: {len(df_valid)} / {len(df_all)}')

In [ ]:
# ── Feature Extraction ───────────────────────────────────────────────────
from dataset.feature.extractor import FeatureExtractor

extractor = FeatureExtractor(
    extract_mel=True,
    extract_mfcc=True,
    extract_f0=True,
    extract_speaker_embed=True,
    n_mels=80,
    n_mfcc=40,
)

df_features = extractor.extract_batch(
    df_valid,
    audio_col='audio_path',
    output_dir=f'{PROCESSED_PATH}/features',
    n_jobs=4,
)
print(f'Features extracted for {len(df_features)} samples')

In [ ]:
# ── Data Augmentation ────────────────────────────────────────────────────
from dataset.augmentation.audio_augment import AudioAugmenter

augmenter = AudioAugmenter(
    apply_spec_augment=True,
    apply_noise=True,
    apply_reverb=True,
    apply_speed_perturb=True,
    augment_factor=2,  # 2x dataset
)

df_augmented = augmenter.augment_dataframe(
    df_features,
    audio_col='audio_path',
    output_dir=f'{PROCESSED_PATH}/augmented',
)
df_combined = pd.concat([df_features, df_augmented], ignore_index=True)
print(f'Combined dataset: {len(df_combined)} samples (orig + augmented)')

In [ ]:
# ── Stratified Split ─────────────────────────────────────────────────────
from dataset.splits.splitter import StratifiedSplitter

splitter = StratifiedSplitter(
    train_ratio=0.80,
    val_ratio=0.10,
    test_ratio=0.10,
    stratify_cols=['province', 'region', 'gender'],
    random_seed=42,
)

splits = splitter.split(df_combined)
print(f'Train: {len(splits["train"])} | Val: {len(splits["val"])} | Test: {len(splits["test"])}')

# Check distribution
for split_name, df_split in splits.items():
    print(f'\n{split_name} region distribution:')
    print(df_split['region'].value_counts())

In [ ]:
# ── Export to HuggingFace Dataset ────────────────────────────────────────
from dataset.export.exporter import DatasetExporter

exporter = DatasetExporter(output_dir=DATASET_EXPORT)
hf_dataset = exporter.export_splits(
    splits,
    dataset_name='vietspeech-vimd',
    push_to_hub=False,  # Set True to upload to HuggingFace Hub
    hub_repo=None,
)

print('\nHuggingFace Dataset created:')
print(hf_dataset)
print(f'\nSaved to: {DATASET_EXPORT}')

## ✓ Dataset Preparation Complete
- Audio validated (wav, 16kHz, mono)
- Features extracted (Mel, MFCC, F0, speaker embeddings)
- Augmented (SpecAugment + noise + reverb + speed)
- Stratified 80/10/10 split by province/region/gender
- Exported to HuggingFace Dataset format

**Next**: Run `02_dialect_identification.ipynb`